In [0]:
# ==========================================
# REAL-TIME JOB PIPELINE CONFIGURATION
# ==========================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType
)

RAW_PATH = "s3://stock-market-raw-pandu/raw/streaming/"

REALTIME_BRONZE_PATH = (
    "s3://stock-market-preprocessed-pandu/"
    "realtime_bronze/"
)

REALTIME_CHECKPOINT_PATH = (
    "s3://stock-market-preprocessed-pandu/"
    "_checkpoints/realtime_job/"
)

CURATED_TRADES_PATH = (
    "s3://stock-market-curated-pandu/trades/"
)

CURATED_BARS_PATH = (
    "s3://stock-market-curated-pandu/bars/"
)


raw_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("event_time", StringType(), True),

    # trade fields
    StructField("price", DoubleType(), True),
    StructField("size", DoubleType(), True),

    # bar fields
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("trade_count", LongType(), True),
    StructField("vwap", DoubleType(), True),

    # common fields
    StructField("source", StringType(), True),
    StructField("source_type", StringType(), True),
    StructField("event_type", StringType(), True)
])

print("Pipeline configuration ready.")

In [0]:
# ==========================================
# READ ONLY NEW RAW S3 FILES
# ==========================================

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")

    # IMPORTANT:
    # Ignore historical files when this NEW stream starts.
    .option("cloudFiles.includeExistingFiles", "false")

    .schema(raw_schema)
    .load(RAW_PATH)
)

print("Auto Loader stream configured.")

In [0]:
# ==========================================
# PROCESS EACH NEW MICRO-BATCH
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


def process_new_batch(batch_df, batch_id):

    if batch_df.isEmpty():
        print(f"Batch {batch_id}: no new records.")
        return

    # ------------------------------
    # CLEAN / STANDARDIZE
    # ------------------------------

    df_clean = (
        batch_df
        .withColumn(
            "event_timestamp",
            F.to_timestamp("event_time")
        )
        .withColumn(
            "event_date",
            F.to_date("event_timestamp")
        )
        .withColumn(
            "symbol",
            F.upper(F.trim("symbol"))
        )
        .dropDuplicates()
    )

    print(
        f"Batch {batch_id}: "
        f"{df_clean.count()} new records"
    )

    # ------------------------------
    # WRITE BRONZE
    # ------------------------------

    (
        df_clean.write
        .format("delta")
        .mode("append")
        .save(REALTIME_BRONZE_PATH)
    )

    # ------------------------------
    # SPLIT TRADES / BARS
    # ------------------------------

    df_trades = (
        df_clean
        .filter(F.col("event_type") == "trade")
    )

    df_bars = (
        df_clean
        .filter(F.col("event_type") == "bar")
    )

    # ------------------------------
    # TRANSFORM TRADES
    # ------------------------------

    if not df_trades.isEmpty():

        trade_window = (
            Window
            .partitionBy("symbol")
            .orderBy("event_timestamp")
        )

        rolling_window = (
            trade_window
            .rowsBetween(-4, 0)
        )

        df_curated_trades = (
            df_trades
            .withColumn(
                "previous_price",
                F.lag("price").over(trade_window)
            )
            .withColumn(
                "price_change",
                F.col("price") -
                F.col("previous_price")
            )
            .withColumn(
                "price_change_pct",
                (
                    F.col("price_change") /
                    F.col("previous_price")
                ) * 100
            )
            .withColumn(
                "rolling_avg_price_5",
                F.avg("price").over(rolling_window)
            )
            .select(
                "symbol",
                "event_timestamp",
                "event_date",
                "price",
                "previous_price",
                "price_change",
                "price_change_pct",
                "rolling_avg_price_5",
                "size",
                "source",
                "source_type"
            )
        )

        (
            df_curated_trades.write
            .mode("append")
            .partitionBy("event_date")
            .parquet(CURATED_TRADES_PATH)
        )

        print(
            f"Batch {batch_id}: "
            f"{df_curated_trades.count()} trades written"
        )

    # ------------------------------
    # TRANSFORM BARS
    # ------------------------------

    if not df_bars.isEmpty():

        df_curated_bars = (
            df_bars
            .withColumn(
                "price_range",
                F.col("high") - F.col("low")
            )
            .withColumn(
                "bar_change",
                F.col("close") - F.col("open")
            )
            .withColumn(
                "bar_change_pct",
                (
                    F.col("bar_change") /
                    F.col("open")
                ) * 100
            )
            .select(
                "symbol",
                "event_timestamp",
                "event_date",
                "open",
                "high",
                "low",
                "close",
                "price_range",
                "bar_change",
                "bar_change_pct",
                "volume",
                "trade_count",
                "vwap",
                "source",
                "source_type"
            )
        )

        (
            df_curated_bars.write
            .mode("append")
            .partitionBy("event_date")
            .parquet(CURATED_BARS_PATH)
        )

        print(
            f"Batch {batch_id}: "
            f"{df_curated_bars.count()} bars written"
        )

In [0]:
# ==========================================
# PROCESS AVAILABLE NEW FILES
# ==========================================

query = (
    df_stream.writeStream
    .foreachBatch(process_new_batch)
    .option(
        "checkpointLocation",
        REALTIME_CHECKPOINT_PATH
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Real-time job iteration completed.")

In [0]:
# ==========================================
# JOB COMPLETION MESSAGE
# ==========================================

print("======================================")
print("Stock market incremental run complete")
print("Waiting for next continuous job run...")
print("======================================")